In [1]:
# importing major libraries 

import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 

import warnings
warnings.filterwarnings('ignore'
)

In [2]:
df = pd.read_csv('dataset_.csv')
df

,age,sex,bmi,children,smoker,region,medical charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520
...,...,...,...,...,...,...,...
1333,50,male,30.970,3,no,northwest,10600.54830
1334,18,female,31.920,0,no,northeast,2205.98080
1335,18,female,36.850,0,no,southeast,1629.83350
1336,21,female,25.800,0,no,southwest,2007.94500


In [3]:
df.sex.unique()
df.region.unique()

array(['southwest', 'southeast', 'northwest', 'northeast'], dtype=object)

In [4]:
# duplicated rows 

df.duplicated().sum()
df.drop_duplicates(inplace=True)

In [5]:
df.shape
df.isnull().sum()

age                0
sex                0
bmi                0
children           0
smoker             0
region             0
medical charges    0
dtype: int64

In [6]:
df.dtypes
df.info()
df.rename({'medical charges':'medical_charges'},axis=1,inplace=True)

<class 'pandas.core.frame.DataFrame'>
Index: 1337 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   age              1337 non-null   int64  
 1   sex              1337 non-null   object 
 2   bmi              1337 non-null   float64
 3   children         1337 non-null   int64  
 4   smoker           1337 non-null   object 
 5   region           1337 non-null   object 
 6   medical charges  1337 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 83.6+ KB


In [7]:
x = df.drop('medical_charges',axis=1)
y = df.medical_charges

from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=1)

print('rows for training',x_train.shape[0])
print('rows for testing',x_test.shape[0])


rows for training 1069
rows for testing 268


In [8]:
df.sex.unique()
l1 =[]
for i in df.columns:
    if i not in df.describe().columns:
        l1.append(i)

print(l1)
for i in l1:
    print(df[i].unique())

['sex', 'smoker', 'region']
['female' 'male']
['yes' 'no']
['southwest' 'southeast' 'northwest' 'northeast']


In [9]:
df

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

trans1 = ColumnTransformer([
    ('sex',OneHotEncoder(sparse_output=False,dtype=int,categories=[['female','male']],drop='first'),['sex']),
    ('smoker',OneHotEncoder(sparse_output=False,dtype=int,categories=[['no','yes']],drop='first'),['smoker']),
    ('region',OneHotEncoder(sparse_output=False,dtype=int,
                            categories=[['southwest','southeast','northwest','northeast']]),['region'])
],remainder='passthrough')

trans1

ColumnTransformer(remainder='passthrough',
                  transformers=[('sex',
                                 OneHotEncoder(categories=[['female', 'male']],
                                               drop='first',
                                               dtype=<class 'int'>,
                                               sparse_output=False),
                                 ['sex']),
                                ('smoker',
                                 OneHotEncoder(categories=[['no', 'yes']],
                                               drop='first',
                                               dtype=<class 'int'>,
                                               sparse_output=False),
                                 ['smoker']),
                                ('region',
                                 OneHotEncoder(categories=[['southwest',
                                                            'southeast',
                                                            'northwest',
                                                            'northeast']],
                                               dtype=<class 'int'>,
                                               sparse_output=False),
                                 ['region'])])

In [10]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.pipeline import Pipeline 

pipe1 = Pipeline([
    ('preprocess',trans1),
    ('estimator',GradientBoostingRegressor(n_estimators=91,
                                           learning_rate=0.05,max_depth=2,
                                           subsample=0.7,max_features=7,min_samples_split=40))
])

# Gradient Boosting – Important Hyperparameters

## 1. n_estimators (Number of Trees)
- Default: 100
- Represents the total number of sequential trees (weak learners)

### Effect:
- Increase:
  - Improves learning capacity
  - May increase accuracy initially
  - Risk of overfitting increases
  - Computational cost increases
- Decrease:
  - Faster training
  - Risk of underfitting

### Best Practice:
- Start with default value
- Increase gradually with proper validation


## 2. learning_rate (Shrinkage)
- Controls the contribution of each tree

### Effect:
- Increase:
  - Faster learning
  - Higher risk of overfitting
- Decrease:
  - Slower learning
  - More stable and generalized model
  - Requires more trees

### Best Practice:
- Use small values (0.01 to 0.1)
- Combine with higher n_estimators


## 3. max_depth (Maximum Depth of Tree)
- Controls the complexity of each individual tree

### Effect:
- Increase:
  - Captures complex patterns
  - Risk of overfitting increases
- Decrease:
  - Simpler model
  - Risk of underfitting

### Best Practice:
- Prefer shallow trees (depth between 3 and 8)


## 4. subsample
- Fraction of training data used for each tree

### Effect:
- Equal to 1:
  - Uses full dataset
  - Higher risk of overfitting
- Less than 1 (e.g., 0.7 to 0.9):
  - Introduces randomness
  - Reduces overfitting
  - Improves generalization

### Best Practice:
- Use values between 0.7 and 0.9


## Rule for Tuning
- If n_estimators increases, learning_rate should decrease

### Reason:
- More trees increase model capacity
- Lower learning rate ensures controlled learning
- Helps maintain balance and prevents overfitting


## Intuition
- n_estimators: Number of learning steps
- learning_rate: Size of each step
- max_depth: Complexity of each step
- subsample: Amount of data used per step

In [11]:
import warnings
warnings.filterwarnings('ignore')
pipe1

Pipeline(steps=[('preprocess',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('sex',
                                                  OneHotEncoder(categories=[['female',
                                                                             'male']],
                                                                drop='first',
                                                                dtype=<class 'int'>,
                                                                sparse_output=False),
                                                  ['sex']),
                                                 ('smoker',
                                                  OneHotEncoder(categories=[['no',
                                                                             'yes']],
                                                                drop='first',
                                                                dtype=<class 'int'>,
                                                                sparse_output=False),
                                                  ['smoker']),
                                                 ('region',
                                                  OneHotEncoder(categories=[['southwest',
                                                                             'southeast',
                                                                             'northwest',
                                                                             'northeast']],
                                                                dtype=<class 'int'>,
                                                                sparse_output=False),
                                                  ['region'])])),
                ('estimator',
                 GradientBoostingRegressor(learning_rate=0.05, max_depth=2,
                                           max_features=7, min_samples_split=40,
                                           n_estimators=91, subsample=0.7))])

In [12]:
pipe1.fit(x_train,y_train)


Pipeline(steps=[('preprocess',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('sex',
                                                  OneHotEncoder(categories=[['female',
                                                                             'male']],
                                                                drop='first',
                                                                dtype=<class 'int'>,
                                                                sparse_output=False),
                                                  ['sex']),
                                                 ('smoker',
                                                  OneHotEncoder(categories=[['no',
                                                                             'yes']],
                                                                drop='first',
                                                                dtype=<class 'int'>,
                                                                sparse_output=False),
                                                  ['smoker']),
                                                 ('region',
                                                  OneHotEncoder(categories=[['southwest',
                                                                             'southeast',
                                                                             'northwest',
                                                                             'northeast']],
                                                                dtype=<class 'int'>,
                                                                sparse_output=False),
                                                  ['region'])])),
                ('estimator',
                 GradientBoostingRegressor(learning_rate=0.05, max_depth=2,
                                           max_features=7, min_samples_split=40,
                                           n_estimators=91, subsample=0.7))])

In [13]:
print('training score',pipe1.score(x_train,y_train))
print('testing score',pipe1.score(x_test,y_test))


training score 0.8676428847107387
testing score 0.8646416085471313
